# 03 — Modelling

We compare two approaches on the precomputed artifacts from `02_Preprocessing.ipynb`:

1. **Feature extraction** — train classic classifiers (XGBoost, Logistic Regression)
   on the frozen DistilBERT `[CLS]` embeddings.
2. **Fine-tuning** — fine-tune the whole DistilBERT model end-to-end.

### Evaluation protocol (no leakage)

* The **validation** set is used for *everything that influences our choices*:
  comparing the baselines and monitoring fine-tuning.
* The **test** set is touched **exactly once**, at the very end, to report the final
  numbers of the selected model. It never informs model selection or tuning.

The original notebook reported its final results on the *validation* set (the same
set used to pick the baseline) and never used the test set — that gives optimistic,
leaked numbers. This notebook fixes that.

## 1. Setup

In [ ]:
# Run once if the packages are missing (e.g. in Colab):
# !pip install -q -r ../requirements.txt

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from datasets import load_from_disk
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

In [ ]:
# Must match 02_Preprocessing.ipynb
MODEL_CKPT = "distilbert-base-uncased"
NUM_LABELS = 6
BATCH_SIZE = 64
EPOCHS = 3
DATA_DIR = Path("..") / "data"
TOKENIZER_COLUMNS = ["input_ids", "attention_mask"]  # DistilBERT model inputs

In [ ]:
# Hidden-state features for the baselines
X_train = np.load(DATA_DIR / "X_train.npy")
X_val = np.load(DATA_DIR / "X_val.npy")
X_test = np.load(DATA_DIR / "X_test.npy")

y_train = np.load(DATA_DIR / "y_train.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

# Tokenized dataset for fine-tuning + the label names
emotions_encoded = load_from_disk(str(DATA_DIR / "emotions_encoded"))
label_names = emotions_encoded["train"].features["label"].names

print("X_train", X_train.shape, "| X_val", X_val.shape, "| X_test", X_test.shape)
print("labels:", label_names)

### Shared evaluation helpers

In [ ]:
def evaluate_model(y_true, y_pred):
    """Accuracy and weighted F1 (weighted because the classes are imbalanced)."""
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="weighted"),
    }


def plot_confusion_matrix(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
    plt.title(f"Normalized confusion matrix — {title}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 2. Feature-extraction baselines

These classifiers are trained on the frozen `[CLS]` embeddings and **compared on
the validation set** to decide which one (if any) we would keep.

### 2.1 Visualize the feature space (UMAP)

A quick sanity check that the embeddings carry class structure. The scaler is fit
on the training features only.

In [ ]:
# !pip install -q umap-learn
import pandas as pd
from umap import UMAP
from sklearn.preprocessing import MinMaxScaler

X_train_scaled = MinMaxScaler().fit_transform(X_train)
mapper = UMAP(n_components=2, metric="cosine").fit(X_train_scaled)

df_emb = pd.DataFrame(mapper.embedding_, columns=["X", "Y"])
df_emb["label"] = y_train
df_emb.head()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
axes = axes.flatten()
cmaps = ["Greys", "Blues", "Oranges", "Reds", "Purples", "Greens"]

for i, (label, cmap) in enumerate(zip(label_names, cmaps)):
    sub = df_emb.query("label == @i")
    axes[i].hexbin(sub["X"], sub["Y"], cmap=cmap, gridsize=20, linewidths=(0,))
    axes[i].set_title(label)
    axes[i].set_xticks([])
    axes[i].set_yticks([])

plt.tight_layout()
plt.show()

### 2.2 Train and compare on the validation set

In [ ]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

baselines = {
    "XGBoost": XGBClassifier(eval_metric="mlogloss"),
    "Logistic Regression": LogisticRegression(max_iter=3000),
}

baseline_val_scores = {}
for name, clf in baselines.items():
    clf.fit(X_train, y_train)
    scores = evaluate_model(y_val, clf.predict(X_val))
    baseline_val_scores[name] = scores
    print(f"{name:>20}: val accuracy={scores['accuracy']:.4f}, val f1={scores['f1']:.4f}")

In [ ]:
for name, clf in baselines.items():
    plot_confusion_matrix(y_val, clf.predict(X_val), label_names, f"{name} (validation)")

In [ ]:
# Pick the better baseline based on validation F1.
best_baseline_name = max(
    baseline_val_scores, key=lambda n: baseline_val_scores[n]["f1"]
)
best_baseline = baselines[best_baseline_name]
print("Selected baseline:", best_baseline_name)

## 3. Fine-tune DistilBERT

End-to-end fine-tuning of `TFAutoModelForSequenceClassification`. We monitor the
validation metrics during training; the test set is still untouched.

In [ ]:
from transformers import TFAutoModelForSequenceClassification

tf_train = emotions_encoded["train"].to_tf_dataset(
    columns=TOKENIZER_COLUMNS, label_cols="label", shuffle=True, batch_size=BATCH_SIZE
)
tf_val = emotions_encoded["validation"].to_tf_dataset(
    columns=TOKENIZER_COLUMNS, label_cols="label", shuffle=False, batch_size=BATCH_SIZE
)
tf_test = emotions_encoded["test"].to_tf_dataset(
    columns=TOKENIZER_COLUMNS, label_cols="label", shuffle=False, batch_size=BATCH_SIZE
)

In [ ]:
tf_model = TFAutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT, num_labels=NUM_LABELS
)
tf_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

In [ ]:
history = tf_model.fit(tf_train, validation_data=tf_val, epochs=EPOCHS)

## 4. Final evaluation on the test set

Now — and only now — we evaluate on the held-out **test** set. We report the
fine-tuned model and, for reference, the selected baseline on the *same* test set.

In [ ]:
# Fine-tuned DistilBERT predictions on the test set.
# tf_test uses shuffle=False, so the order matches y_test from 02_Preprocessing.
test_logits = tf_model.predict(tf_test).logits
y_pred_finetuned = np.argmax(test_logits, axis=1)

# Selected baseline on the same test set.
y_pred_baseline = best_baseline.predict(X_test)

In [ ]:
results = pd.DataFrame(
    {
        f"{best_baseline_name} (features)": evaluate_model(y_test, y_pred_baseline),
        "DistilBERT (fine-tuned)": evaluate_model(y_test, y_pred_finetuned),
    }
).T
results

In [ ]:
plot_confusion_matrix(y_test, y_pred_finetuned, label_names, "DistilBERT, test set")

### Inspect some predictions

The original notebook mapped class ids to names by indexing a dataframe column by
row position, which produced wrong labels. Here we map each class id through
`label_names` directly.

In [ ]:
test_texts = emotions_encoded["test"]["text"]

for i in range(20):
    predicted = label_names[y_pred_finetuned[i]]
    actual = label_names[y_test[i]]
    mark = "OK " if predicted == actual else "XX "
    print(f"{mark} predicted={predicted:<9} actual={actual:<9} | {test_texts[i]}")

## 5. Conclusion

* Fine-tuning DistilBERT clearly outperforms the feature-extraction baselines.
* All final numbers above come from the **test set**, which was never used for
  model selection — so they are an honest estimate of generalization.
* The confusion matrix shows the remaining errors concentrate on the rare,
  semantically close classes (*love* vs *joy*, *surprise* vs *fear*), consistent
  with the class imbalance seen in `01_EDA.ipynb`.